In [3]:
# ═══════════════════════════════════════════════════════════════════════════════
# NOTEBOOK 02 — AI Governance Evaluation (G1–G4)   (REVISED, self-contained)
# ═══════════════════════════════════════════════════════════════════════════════
# Loads only the primitive artefacts saved by Notebook 01 (df_final, best_models,
# best_algo_name, all_results, config) and computes CV stability, temporal
# validation, SHAP+HHI, and the four-pillar scorecard internally — so the
# governance logic is fully reproducible from this notebook alone.
#
# Revisions (addressing referee comments):
#   [G1-DUAL]   G1 Fairness reported two ways — G1-raw (ΔStageAcc+ΔRMSE, base-rate
#               confounded) and G1-corrected (ΔBalancedAcc+ΔCohenKappa, invariant).
#   [G3-CAVEAT] HHI reported but flagged: low HHI ≠ better transparency.
#   [WEIGHTS]   Weights exposed; one-line sensitivity sweep included.
#   [PATHS]     results/{tables,figures,artifacts}; no absolute paths printed.
#   [FIGURES]   seaborn grayscale; no captions; dpi=600; saved as BOTH png + pdf.
# ═══════════════════════════════════════════════════════════════════════════════

import warnings; warnings.filterwarnings("ignore")
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import shap
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (mean_squared_error, mean_absolute_error, r2_score,
                             balanced_accuracy_score, cohen_kappa_score, f1_score)
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
import lightgbm as lgb
import xgboost as xgb

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger(__name__)

SEED = 42; DPI = 600
np.random.seed(SEED)

# ── Grayscale publication style (seaborn) ─────────────────────────────────────
sns.set_theme(style="whitegrid", context="paper")
sns.set_palette("Greys")
plt.rcParams.update({
    "font.family":"DejaVu Sans", "font.size":11, "axes.unicode_minus":False,
    "figure.dpi":150, "savefig.dpi":DPI,
    "axes.edgecolor":"0.2", "axes.linewidth":0.8,
    "grid.color":"0.85", "grid.linewidth":0.6,
})

def save_fig(fig, name):
    """Save a figure as BOTH png and pdf at dpi=600, no caption, tight bbox."""
    for ext in ("png", "pdf"):
        fig.savefig(FIG_DIR / f"{name}.{ext}", dpi=DPI, bbox_inches="tight")
    plt.close(fig)

# ── Project paths (robust; no absolute path printed) ──────────────────────────
def find_project_root(start: Path = Path.cwd()) -> Path:
    for p in [start, *start.parents]:
        if (p / "data").is_dir() and (p / "results").is_dir():
            return p
    return start.parent if start.name == "notebooks" else start

ROOT      = find_project_root()
FIG_DIR   = ROOT / "results" / "figures"
TABLE_DIR = ROOT / "results" / "tables"
ART_DIR   = ROOT / "results" / "artifacts"
for d in (FIG_DIR, TABLE_DIR, ART_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# 1. LOAD PRIMITIVES FROM NOTEBOOK 01
# ═══════════════════════════════════════════════════════════════════════════════
def _load_df_final():
    p_parq = ART_DIR / "df_final.parquet"; p_csv = ART_DIR / "df_final.csv"
    if p_parq.exists():
        try:    return pd.read_parquet(p_parq)
        except Exception: pass
    return pd.read_csv(p_csv)

df_final       = _load_df_final()
best_models    = joblib.load(ART_DIR / "best_models.pkl")
best_algo_name = joblib.load(ART_DIR / "best_algo_name.pkl")
all_results    = joblib.load(ART_DIR / "all_results.pkl")
cfg            = joblib.load(ART_DIR / "config.pkl")

X_FEATURES   = cfg["X_FEATURES"]; NUM_FEATURES = cfg["NUM_FEATURES"]
CAT_FEATURES = cfg["CAT_FEATURES"]
GROUP_CONFIG = cfg["GROUP_CONFIG"]; GROUP_LABELS = cfg["GROUP_LABELS"]
STAGE_ORDER  = cfg["STAGE_ORDER"]
ALGORITHMS   = ["LR","Ridge","RF","LGBM","XGB","MLP"]

# Guard: after a CSV round-trip some numeric columns can load as object dtype,
# which later breaks XGBoost's SHAP explainer. Coerce all model inputs + FPG to
# float once, up front. (GlucoseStage/text columns are left untouched.)
_num_cols = [c for c in (X_FEATURES + ["FPG", "AgeGroup", "Sex", "SurveyYear"])
             if c in df_final.columns]
df_final[_num_cols] = df_final[_num_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0).astype("float64")
log.info("Coerced %d numeric columns to float64.", len(_num_cols))

def assign_glucose_stage(fpg):
    if fpg < 100:  return "normal"
    if fpg < 126:  return "ifg"
    return "diabetes"
def stage_of(vals):
    return pd.Series([assign_glucose_stage(v) for v in np.asarray(vals)], dtype=object)

log.info("Primitives loaded. df_final: %s", df_final.shape)

def group_holdout(grp):
    c = GROUP_CONFIG[grp]
    df_g = df_final[(df_final["AgeGroup"]==c["age_group"]) &
                    (df_final["Sex"]==c["sex_code"])].copy()
    X, y = df_g[X_FEATURES], df_g["FPG"]
    Xtr, Xv, ytr, yv = train_test_split(X, y, test_size=0.20, random_state=SEED)
    return df_g, Xtr, Xv, ytr, yv

# ═══════════════════════════════════════════════════════════════════════════════
# 2. THRESHOLDS, WEIGHTS & SCORING UTILITIES
# ═══════════════════════════════════════════════════════════════════════════════
THRESHOLDS = {
    "G1_delta_rmse":     {"ok":3.0,  "caution":6.0,  "cite":"Obermeyer et al. (2019)"},
    "G1_delta_stageacc": {"ok":0.10, "caution":0.20, "cite":"Chouldechova (2017), raw"},
    "G1_delta_balacc":   {"ok":0.10, "caution":0.20, "cite":"Hardt et al. (2016), balanced"},
    "G1_delta_kappa":    {"ok":0.15, "caution":0.30, "cite":"Cohen (1960); chance-corrected"},
    "G2_cv_gap":         {"ok":1.0,  "caution":2.0,  "cite":"Varma & Simon (2006)"},
    "G2_temporal":       {"ok":1.5,  "caution":3.5,  "cite":"Nestor et al. (2019)"},
    "G2_perturbation":   {"ok":1.0,  "caution":2.5,  "cite":"Ghorbani & Zou (2019)"},
    "G3_hhi":            {"ok":0.18, "caution":0.25, "cite":"Lundberg et al. (2020)"},
    "G4_model_card":     {"ok":0.90, "caution":0.70, "cite":"Mitchell et al. (2019)"},
}
WEIGHTS = {"g1":0.35, "g2":0.25, "g3":0.20, "g4":0.20}

def verdict(value, key, low_is_good=True):
    t = THRESHOLDS[key]
    if low_is_good:
        if value <= t["ok"]:      return "OK"
        if value <= t["caution"]: return "Caution"
        return "Risk"
    else:
        if value >= t["ok"]:      return "OK"
        if value >= t["caution"]: return "Caution"
        return "Risk"

def normalise(value, ok_val, risk_val, low_is_good=True, floor=0.05):
    if low_is_good:
        if value <= ok_val:   return 1.0
        if value >= risk_val: return floor
        return floor + (1.0-floor)*(1.0-(value-ok_val)/(risk_val-ok_val))
    else:
        if value >= ok_val:   return 1.0
        if value <= risk_val: return floor
        return floor + (1.0-floor)*((value-risk_val)/(ok_val-risk_val))

log.info("Thresholds, weights, utilities defined.")

# ═══════════════════════════════════════════════════════════════════════════════
# 3. G1 — FAIRNESS  (dual: raw + base-rate-corrected)   ★ core revision
# ═══════════════════════════════════════════════════════════════════════════════
g1_rows = []
for grp in GROUP_CONFIG:
    model = best_models.get(grp)
    if model is None: continue
    _, _, Xv, _, yv = group_holdout(grp)
    yp = model.predict(Xv)

    rmse = float(np.sqrt(mean_squared_error(yv, yp)))
    mae  = float(mean_absolute_error(yv, yp))
    r2   = float(r2_score(yv, yp))
    bias = float(np.mean(yp - yv.values))

    st, sp = stage_of(yv.values), stage_of(yp)
    stage_acc = float((st.values==sp.values).mean())
    maj_cls   = st.value_counts().idxmax()
    maj_base  = float((st.values==maj_cls).mean())
    bal_acc   = float(balanced_accuracy_score(st, sp))
    kappa     = float(cohen_kappa_score(st, sp, labels=STAGE_ORDER))
    macro_f1  = float(f1_score(st, sp, average="macro", labels=STAGE_ORDER))

    g1_rows.append({"Group":GROUP_LABELS[grp], "_key":grp, "Algorithm":best_algo_name[grp],
        "n_test":int(len(yv)), "RMSE":round(rmse,3), "MAE":round(mae,3), "R2":round(r2,3),
        "Bias":round(bias,3), "StageAcc":round(stage_acc,3), "MajBaseline":round(maj_base,3),
        "Improvement":round(stage_acc-maj_base,3), "BalancedAcc":round(bal_acc,3),
        "CohenKappa":round(kappa,3), "MacroF1":round(macro_f1,3)})

g1_df = pd.DataFrame(g1_rows)

d_rmse     = round(g1_df["RMSE"].max()        - g1_df["RMSE"].min(), 3)
d_stageacc = round(g1_df["StageAcc"].max()    - g1_df["StageAcc"].min(), 3)
d_balacc   = round(g1_df["BalancedAcc"].max() - g1_df["BalancedAcc"].min(), 3)
d_kappa    = round(g1_df["CohenKappa"].max()  - g1_df["CohenKappa"].min(), 3)
d_improve  = round(g1_df["Improvement"].max() - g1_df["Improvement"].min(), 3)

g1_summary = pd.DataFrame([
    {"Metric":"Δ-RMSE",            "Value":d_rmse,     "Verdict":verdict(d_rmse,"G1_delta_rmse"),     "Type":"raw"},
    {"Metric":"Δ-StageAccuracy",   "Value":d_stageacc, "Verdict":verdict(d_stageacc,"G1_delta_stageacc"),"Type":"raw (confounded)"},
    {"Metric":"Δ-BalancedAccuracy","Value":d_balacc,   "Verdict":verdict(d_balacc,"G1_delta_balacc"),  "Type":"corrected"},
    {"Metric":"Δ-CohenKappa",      "Value":d_kappa,    "Verdict":verdict(d_kappa,"G1_delta_kappa"),    "Type":"corrected"},
    {"Metric":"Δ-Improvement",     "Value":d_improve,  "Verdict":"—",                                  "Type":"corrected (context)"},
])

print("\n── G1 per-group metrics ──")
print(g1_df.drop(columns=["_key"]).to_string(index=False))
print("\n── G1 disparities: raw vs base-rate-corrected ──")
print(g1_summary.to_string(index=False))

g1_df.drop(columns=["_key"]).to_csv(TABLE_DIR / "table_g1_fairness.csv", index=False, encoding="utf-8")
g1_summary.to_csv(TABLE_DIR / "table_g1_summary.csv", index=False, encoding="utf-8")

s_g1_raw       = round((normalise(d_rmse,3.0,6.0) + normalise(d_stageacc,0.10,0.20))/2, 4)
s_g1_corrected = round((normalise(d_balacc,0.10,0.20) + normalise(d_kappa,0.15,0.30))/2, 4)
log.info("G1-raw score = %.3f | G1-corrected score = %.3f", s_g1_raw, s_g1_corrected)

# ═══════════════════════════════════════════════════════════════════════════════
# 4. G2 — ROBUSTNESS  (CV stability + temporal + perturbation)
# ═══════════════════════════════════════════════════════════════════════════════
def refit(algo, Xtr, ytr):
    if   algo=="LR":    m = LinearRegression()
    elif algo=="Ridge": m = Ridge(random_state=SEED)
    elif algo=="RF":    m = RandomForestRegressor(n_estimators=300, max_depth=8, random_state=SEED, n_jobs=-1)
    elif algo=="LGBM":  m = lgb.LGBMRegressor(n_estimators=300, max_depth=5, learning_rate=0.05, random_state=SEED, n_jobs=-1, verbose=-1)
    elif algo=="XGB":   m = xgb.XGBRegressor(n_estimators=300, max_depth=5, learning_rate=0.05, random_state=SEED, tree_method="hist", verbosity=0)
    elif algo=="MLP":   m = MLPRegressor(hidden_layer_sizes=(128,128), max_iter=300, random_state=SEED)
    m.fit(Xtr, ytr); return m

# G2-a: 5-fold CV stability
g2a_rows = []
for grp in GROUP_CONFIG:
    algo = best_algo_name[grp]
    df_g, Xtr, Xv, ytr, yv = group_holdout(grp)
    ho_rmse = float(np.sqrt(mean_squared_error(yv, best_models[grp].predict(Xv))))
    X, y = df_g[X_FEATURES], df_g["FPG"]
    kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
    fold_rmses = []
    for tr, va in kf.split(X):
        m = refit(algo, X.iloc[tr], y.iloc[tr])
        fold_rmses.append(np.sqrt(mean_squared_error(y.iloc[va], m.predict(X.iloc[va]))))
    cv_mean, cv_std = float(np.mean(fold_rmses)), float(np.std(fold_rmses))
    gap = round(abs(cv_mean - ho_rmse), 4)
    g2a_rows.append({"Group":GROUP_LABELS[grp], "Algorithm":algo,
        "HO_RMSE":round(ho_rmse,3), "CV_RMSE_mean":round(cv_mean,3),
        "CV_RMSE_std":round(cv_std,3), "Gap":gap, "G2a_verdict":verdict(gap,"G2_cv_gap")})
g2a_df = pd.DataFrame(g2a_rows)

# G2-b: temporal validation (train earlier cycles → test most recent)
years = sorted(pd.to_numeric(df_final["SurveyYear"], errors="coerce").dropna().unique())
YEAR_TEST  = years[-1] if years else None
YEAR_TRAIN = years[:-1] if len(years) > 1 else years
g2b_rows = []
if YEAR_TEST is not None and len(YEAR_TRAIN) > 0:
    for grp in GROUP_CONFIG:
        algo = best_algo_name[grp]; c = GROUP_CONFIG[grp]
        df_g = df_final[(df_final["AgeGroup"]==c["age_group"]) & (df_final["Sex"]==c["sex_code"])]
        tr = df_g[df_g["SurveyYear"].isin(YEAR_TRAIN)]
        te = df_g[df_g["SurveyYear"]==YEAR_TEST]
        if len(tr) < 50 or len(te) < 20:
            continue
        m = refit(algo, tr[X_FEATURES], tr["FPG"])
        _, _, Xv_g, _, yv_g = group_holdout(grp)
        ho_rmse = float(np.sqrt(mean_squared_error(yv_g, m.predict(Xv_g))))
        te_rmse = float(np.sqrt(mean_squared_error(te["FPG"], m.predict(te[X_FEATURES]))))
        delta = round(te_rmse - ho_rmse, 3)
        g2b_rows.append({"Group":GROUP_LABELS[grp], "Algorithm":algo,
            "HO_RMSE":round(ho_rmse,3), "Temporal_RMSE":round(te_rmse,3), "Delta":delta,
            "Direction":("degradation" if delta>0 else "stable/improve"),
            "G2b_verdict":verdict(max(delta,0.0),"G2_temporal")})
g2b_df = pd.DataFrame(g2b_rows)

# G2-c: feature perturbation (±10% Gaussian noise, 5 repeats)
g2c_rows = []; rng = np.random.RandomState(SEED)
for grp in GROUP_CONFIG:
    model = best_models.get(grp)
    if model is None: continue
    _, _, Xv, _, yv = group_holdout(grp)
    base = float(np.sqrt(mean_squared_error(yv, model.predict(Xv))))
    drops = []
    for _ in range(5):
        Xn = Xv.copy()
        for col in Xn.columns:
            s = Xn[col].std()
            if s > 0: Xn[col] += rng.normal(0, s*0.10, len(Xn))
        drops.append(abs(float(np.sqrt(mean_squared_error(yv, model.predict(Xn)))) - base))
    md = round(float(np.mean(drops)), 4)
    g2c_rows.append({"Group":GROUP_LABELS[grp], "Algorithm":best_algo_name[grp],
        "RMSE_base":round(base,3), "RMSE_drop_mean":md, "RMSE_drop_std":round(float(np.std(drops)),4),
        "G2c_verdict":verdict(md,"G2_perturbation")})
g2c_df = pd.DataFrame(g2c_rows)

print("\n── G2-a: CV stability ──");        print(g2a_df.to_string(index=False))
print("\n── G2-b: temporal validation ──"); print(g2b_df.to_string(index=False) if len(g2b_df) else "  (insufficient per-year n)")
print("\n── G2-c: perturbation ──");         print(g2c_df.to_string(index=False))

pd.concat([g2a_df.assign(dimension="G2a_CV"),
           g2b_df.assign(dimension="G2b_Temporal") if len(g2b_df) else pd.DataFrame(),
           g2c_df.assign(dimension="G2c_Perturbation")], ignore_index=True
          ).to_csv(TABLE_DIR / "table_g2_robustness.csv", index=False, encoding="utf-8")

s_g2a = normalise(g2a_df["Gap"].mean(), 1.0, 2.0)
s_g2c = normalise(g2c_df["RMSE_drop_mean"].mean(), 1.0, 2.5)
if len(g2b_df):
    deg = g2b_df.loc[g2b_df["Delta"]>0, "Delta"]
    s_g2b = normalise(deg.max() if len(deg) else 0.0, 1.5, 3.5)
else:
    s_g2b = 1.0
s_g2 = round((s_g2a + s_g2b + s_g2c)/3, 4)
log.info("G2 score = %.3f (a=%.2f b=%.2f c=%.2f)", s_g2, s_g2a, s_g2b, s_g2c)

# ═══════════════════════════════════════════════════════════════════════════════
# 5. G3 — TRANSPARENCY  (SHAP HHI, bootstrapped) + explicit caveat  [G3-CAVEAT]
# ═══════════════════════════════════════════════════════════════════════════════
def compute_hhi_bootstrap(shap_vals, n_boot=1000, ci=0.95):
    def _hhi(sv):
        m = np.mean(np.abs(sv), axis=0); tot = m.sum()
        if tot == 0: return 0.0
        s = m/tot; return float(np.sum(s**2))
    point = _hhi(shap_vals)
    rng_ = np.random.RandomState(SEED); n = shap_vals.shape[0]; boots = []
    for _ in range(n_boot):
        idx = rng_.randint(0, n, n)
        boots.append(_hhi(shap_vals[idx]))
    lo = float(np.percentile(boots, (1-ci)/2*100))
    hi = float(np.percentile(boots, (1+ci)/2*100))
    return {"HHI":round(point,4), "CI_lo":round(lo,4), "CI_hi":round(hi,4)}

g3_rows = []; shap_store = {}
for grp in GROUP_CONFIG:
    model = best_models.get(grp); algo = best_algo_name.get(grp)
    if model is None: continue
    _, _, Xv, _, _ = group_holdout(grp)
    # Force a clean float matrix. Some columns can arrive as object dtype after a
    # CSV round-trip; any stray '[...]' strings become NaN → 0.0 here.
    Xv_num = Xv.apply(pd.to_numeric, errors="coerce").fillna(0.0).astype("float64")
    Xarr = np.ascontiguousarray(Xv_num.values, dtype=np.float64)  # pure ndarray

    hhi = {"HHI": np.nan, "CI_lo": np.nan, "CI_hi": np.nan}
    try:
        if algo in ("RF", "LGBM", "XGB"):
            expl = shap.TreeExplainer(model)
            # try DataFrame → ndarray → additivity-off, in order
            for attempt in ("df", "arr", "arr_noadd"):
                try:
                    if attempt == "df":
                        sv = np.asarray(expl.shap_values(Xv_num))
                    elif attempt == "arr":
                        sv = np.asarray(expl.shap_values(Xarr))
                    else:
                        sv = np.asarray(expl.shap_values(Xarr, check_additivity=False))
                    break
                except Exception as inner:
                    last_err = inner
            else:
                raise last_err
        else:
            bg = shap.sample(Xv_num, min(50, len(Xv_num)), random_state=SEED)
            expl = shap.KernelExplainer(model.predict, bg)
            sv = np.asarray(expl.shap_values(Xv_num.iloc[:min(100, len(Xv_num))], nsamples=100))

        if sv.ndim == 3:
            sv = sv[..., 0] if sv.shape[-1] == 1 else sv.reshape(sv.shape[0], -1)
        shap_store[grp] = {"shap": sv, "X": Xv_num.iloc[:sv.shape[0]].copy()}
        hhi = compute_hhi_bootstrap(sv)
    except Exception as e:
        log.warning("SHAP failed for %s (%s): %s", grp, algo, e)

    g3_rows.append({"Group":GROUP_LABELS[grp], "Algorithm":algo,
        "SHAP_HHI":hhi["HHI"], "CI_95_lower":hhi["CI_lo"], "CI_95_upper":hhi["CI_hi"],
        "G3_verdict":verdict(hhi["HHI"],"G3_hhi") if not np.isnan(hhi["HHI"]) else "n/a"})
g3_df = pd.DataFrame(g3_rows)

g3_df = g3_df.merge(g1_df[["Group","BalancedAcc"]], on="Group", how="left")
g3_df["Caveat"] = np.where(
    (g3_df["SHAP_HHI"] < 0.18) & (g3_df["BalancedAcc"] < 0.42),
    "diffuse attribution but low accuracy — not genuine transparency", "")
print("\n── G3: Transparency (SHAP HHI) with diffusion-without-accuracy caveat ──")
print(g3_df.to_string(index=False))
g3_df.to_csv(TABLE_DIR / "table_g3_transparency.csv", index=False, encoding="utf-8")

s_g3 = normalise(g3_df["SHAP_HHI"].mean(skipna=True), 0.18, 0.25)
log.info("G3 score = %.3f (mean HHI = %.3f)", s_g3, g3_df["SHAP_HHI"].mean(skipna=True))

# ═══════════════════════════════════════════════════════════════════════════════
# 6. G4 — ACCOUNTABILITY (Model Card checklist)
# ═══════════════════════════════════════════════════════════════════════════════
GITHUB_URL = ""   # set to public repo URL when available
MODEL_CARD = {
    "Intended purpose documented": True,
    "Training data source and period stated": True,
    "Medicated-diabetic exclusion criterion stated": True,
    "Performance metrics reported (RMSE/MAE/R2)": True,
    "Base-rate-corrected fairness metrics reported": True,
    "Limitations and failure modes disclosed": True,
    "Disaggregated performance by group": True,
    "SHAP attribution available": True,
    "DiCE counterfactual paths provided": True,
    "LLM personalised reports generated": True,
    "Code publicly available (GitHub)": bool(GITHUB_URL),
    "Human-in-the-loop procedure defined": False,   # gap
    "Privacy Impact Assessment (PIA) conducted": False,   # gap
}
total = len(MODEL_CARD); fulfilled = sum(MODEL_CARD.values())
mc_score = round(fulfilled/total, 4)
mc_verdict = verdict(mc_score, "G4_model_card", low_is_good=False)
print(f"\n── G4: Accountability — Model Card {fulfilled}/{total} = {mc_score:.2f} [{mc_verdict}] ──")
for k, v in MODEL_CARD.items():
    print(f"  [{'x' if v else ' '}] {k}")
s_g4 = normalise(mc_score, 0.90, 0.70, low_is_good=False)
pd.DataFrame([{"item":k, "fulfilled":v} for k,v in MODEL_CARD.items()]
             ).to_csv(TABLE_DIR / "table_g4_accountability.csv", index=False, encoding="utf-8")

# ═══════════════════════════════════════════════════════════════════════════════
# 7. COMPOSITE SCORECARD  (default: G1-corrected; G1-raw reported alongside)
# ═══════════════════════════════════════════════════════════════════════════════
def composite_of(s_g1):
    return round(WEIGHTS["g1"]*s_g1 + WEIGHTS["g2"]*s_g2 + WEIGHTS["g3"]*s_g3 + WEIGHTS["g4"]*s_g4, 4)
comp_corrected = composite_of(s_g1_corrected)
comp_raw       = composite_of(s_g1_raw)
def grade(c):
    return ("Green — Deploy" if c>=0.80 else "Amber — Conditional" if c>=0.60 else "Red — Do not deploy")

scorecard = pd.DataFrame([
    {"Pillar":"G1 Fairness (corrected)","Weight":WEIGHTS["g1"],"Score":s_g1_corrected,
     "Verdict":verdict(d_balacc,"G1_delta_balacc")},
    {"Pillar":"  G1 Fairness (raw, ref.)","Weight":"—","Score":s_g1_raw,
     "Verdict":verdict(d_stageacc,"G1_delta_stageacc")},
    {"Pillar":"G2 Robustness","Weight":WEIGHTS["g2"],"Score":s_g2,
     "Verdict":"OK" if s_g2>=0.8 else "Caution" if s_g2>=0.5 else "Risk"},
    {"Pillar":"G3 Transparency","Weight":WEIGHTS["g3"],"Score":s_g3,
     "Verdict":g3_df["G3_verdict"].mode()[0] if len(g3_df) else "n/a"},
    {"Pillar":"G4 Accountability","Weight":WEIGHTS["g4"],"Score":s_g4,"Verdict":mc_verdict},
    {"Pillar":"COMPOSITE (G1-corrected)","Weight":1.00,"Score":comp_corrected,"Verdict":grade(comp_corrected)},
    {"Pillar":"COMPOSITE (G1-raw, ref.)","Weight":"—","Score":comp_raw,"Verdict":grade(comp_raw)},
])
print("\n── Four-pillar governance scorecard ──")
print(scorecard.to_string(index=False))
scorecard.to_csv(TABLE_DIR / "table_governance_scorecard.csv", index=False, encoding="utf-8")
log.info("Composite (G1-corrected) = %.3f [%s]", comp_corrected, grade(comp_corrected))
log.info("Composite (G1-raw)       = %.3f [%s]", comp_raw, grade(comp_raw))

# Weight sensitivity sweep
sens_rows = []
for w1 in [0.25, 0.30, 0.35, 0.40, 0.45]:
    rem = 1 - w1
    w2, w3, w4 = rem*0.25/0.65, rem*0.20/0.65, rem*0.20/0.65
    c = round(w1*s_g1_corrected + w2*s_g2 + w3*s_g3 + w4*s_g4, 4)
    sens_rows.append({"w_G1":w1, "Composite_corrected":c, "Grade":grade(c)})
sens_df = pd.DataFrame(sens_rows)
print("\n── Weight sensitivity (G1 weight swept) ──")
print(sens_df.to_string(index=False))
sens_df.to_csv(TABLE_DIR / "table_weight_sensitivity.csv", index=False, encoding="utf-8")

# ═══════════════════════════════════════════════════════════════════════════════
# 8. FIGURES  (seaborn grayscale; no captions; dpi=600; png + pdf)
# ═══════════════════════════════════════════════════════════════════════════════
labels = [GROUP_LABELS[g] for g in GROUP_CONFIG]

# --- Figure 1: G1 raw vs corrected (the key referee-facing plot) ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# (a) StageAcc vs Majority baseline — long-form for seaborn
sa = [g1_df.loc[g1_df["Group"]==l,"StageAcc"].values[0] for l in labels]
mb = [g1_df.loc[g1_df["Group"]==l,"MajBaseline"].values[0] for l in labels]
dfa = pd.DataFrame({"Group":labels*2,
                    "Value":sa+mb,
                    "Metric":["Model Stage Accuracy"]*len(labels)+["Majority baseline"]*len(labels)})
sns.barplot(data=dfa, x="Group", y="Value", hue="Metric",
            palette=["0.30","0.75"], edgecolor="black", ax=axes[0])
axes[0].set_ylim(0,1); axes[0].set_ylabel("Accuracy"); axes[0].set_xlabel("")
axes[0].set_xticklabels(labels, rotation=30, ha="right", fontsize=8)
axes[0].legend(fontsize=8, loc="lower left", title="")

# (b) Balanced accuracy vs Cohen's kappa
ba = [g1_df.loc[g1_df["Group"]==l,"BalancedAcc"].values[0] for l in labels]
kp = [g1_df.loc[g1_df["Group"]==l,"CohenKappa"].values[0] for l in labels]
dfb = pd.DataFrame({"Group":labels*2,
                    "Value":ba+kp,
                    "Metric":["Balanced accuracy"]*len(labels)+["Cohen's kappa"]*len(labels)})
sns.barplot(data=dfb, x="Group", y="Value", hue="Metric",
            palette=["0.30","0.75"], edgecolor="black", ax=axes[1])
axes[1].set_ylim(0,1); axes[1].set_ylabel("Score"); axes[1].set_xlabel("")
axes[1].set_xticklabels(labels, rotation=30, ha="right", fontsize=8)
axes[1].legend(fontsize=8, loc="upper right", title="")
plt.tight_layout()
save_fig(fig, "fig_g1_raw_vs_corrected")

# --- Figure 2: governance radar (grayscale) ---
cats = ["G1\nFairness","G2\nRobustness","G3\nTransparency","G4\nAccountability"]
scores = [s_g1_corrected, s_g2, s_g3, s_g4]
N = len(cats); ang = np.linspace(0, 2*np.pi, N, endpoint=False).tolist(); ang += ang[:1]
sc = scores + [scores[0]]
fig, ax = plt.subplots(figsize=(7,7), subplot_kw=dict(polar=True))
ax.plot(ang, sc, "o-", color="0.15", lw=2.5, ms=8)
ax.fill(ang, sc, color="0.15", alpha=0.15)
ax.plot(ang, [0.80]*(N+1), ls="--", color="0.45", lw=1.2, label="Deploy threshold (0.80)")
ax.plot(ang, [0.60]*(N+1), ls=":",  color="0.45", lw=1.2, label="Conditional threshold (0.60)")
ax.set_xticks(ang[:-1]); ax.set_xticklabels(cats, fontsize=11, fontweight="bold")
ax.set_ylim(0,1); ax.set_yticks([0.2,0.4,0.6,0.8,1.0])
ax.set_yticklabels(["0.2","0.4","0.6","0.8","1.0"], fontsize=8, color="0.4")
for a, s in zip(ang[:-1], scores):
    ax.text(a, s+0.09, f"{s:.2f}", ha="center", va="center", fontsize=10, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="0.3", alpha=0.9))
ax.legend(fontsize=9, loc="lower right", bbox_to_anchor=(1.25,-0.08))
plt.tight_layout(pad=2.5)
save_fig(fig, "fig_governance_radar")

# --- Figure 3: RMSE heatmap (seaborn grayscale) ---
rmse_mat = pd.DataFrame(index=list(GROUP_CONFIG), columns=ALGORITHMS, dtype=float)
for grp in GROUP_CONFIG:
    for algo in ALGORITHMS:
        m = all_results.get(grp, {}).get(algo)
        rmse_mat.loc[grp, algo] = m["RMSE"] if m else np.nan
rmse_mat.index = [GROUP_LABELS[g] for g in rmse_mat.index]
fig, ax = plt.subplots(figsize=(11,5))
sns.heatmap(rmse_mat.astype(float), annot=True, fmt=".2f", cmap="Greys",
            linewidths=0.5, linecolor="white",
            cbar_kws={"label":"RMSE (mg/dL)", "shrink":0.8}, ax=ax)
for i, grp in enumerate(GROUP_CONFIG):
    ba_ = best_algo_name.get(grp)
    if ba_ in ALGORITHMS:
        ax.text(ALGORITHMS.index(ba_)+0.5, i+0.22, "*", ha="center", va="center",
                fontsize=16, color="black")
ax.set_xlabel("Algorithm"); ax.set_ylabel("Demographic Group")
plt.tight_layout()
save_fig(fig, "fig_rmse_heatmap")

log.info("Figures saved (png + pdf, dpi=%d) to results/figures/", DPI)

# ═══════════════════════════════════════════════════════════════════════════════
# 9. SAVE CONSOLIDATED GOVERNANCE ARTEFACTS
# ═══════════════════════════════════════════════════════════════════════════════
joblib.dump({"g1_df":g1_df, "g1_summary":g1_summary,
             "s_g1_raw":s_g1_raw, "s_g1_corrected":s_g1_corrected,
             "s_g2":s_g2, "s_g3":s_g3, "s_g4":s_g4,
             "comp_corrected":comp_corrected, "comp_raw":comp_raw,
             "g2a_df":g2a_df, "g2b_df":g2b_df, "g2c_df":g2c_df,
             "g3_df":g3_df, "shap_store":shap_store, "scorecard":scorecard,
             "sens_df":sens_df, "WEIGHTS":WEIGHTS, "THRESHOLDS":THRESHOLDS},
            ART_DIR / "governance_results.pkl")
log.info("Notebook 02 complete. Composite (corrected) = %.3f", comp_corrected)

2026-09-01 15:11:02,355 | INFO | Coerced 35 numeric columns to float64.
2026-09-01 15:11:02,356 | INFO | Primitives loaded. df_final: (27934, 36)
2026-09-01 15:11:02,359 | INFO | Thresholds, weights, utilities defined.
2026-09-01 15:11:02,681 | INFO | G1-raw score = 0.050 | G1-corrected score = 0.680



── G1 per-group metrics ──
             Group Algorithm  n_test   RMSE    MAE    R2   Bias  StageAcc  MajBaseline  Improvement  BalancedAcc  CohenKappa  MacroF1
        Young Male      LGBM     613 11.239  7.017 0.091  0.922     0.790        0.783        0.007        0.407       0.259    0.415
      Young Female       XGB     719 14.381  7.164 0.159 -0.530     0.880        0.890       -0.010        0.402       0.256    0.411
  Middle-aged Male       XGB    1096 25.932 15.331 0.061 -0.246     0.462        0.483       -0.021        0.380       0.102    0.307
Middle-aged Female       XGB    1461 21.189 10.977 0.119  0.002     0.625        0.656       -0.031        0.437       0.248    0.424
      Elderly Male        RF     739 26.296 16.589 0.019 -0.486     0.441        0.438        0.003        0.341       0.008    0.220
    Elderly Female       XGB     961 24.417 15.526 0.043 -0.955     0.417        0.504       -0.086        0.362       0.058    0.280

── G1 disparities: raw vs base-ra

2026-09-01 15:11:38,635 | INFO | G2 score = 0.481 (a=0.39 b=0.05 c=1.00)



── G2-a: CV stability ──
             Group Algorithm  HO_RMSE  CV_RMSE_mean  CV_RMSE_std    Gap G2a_verdict
        Young Male      LGBM   11.239        16.309        3.305 5.0699        Risk
      Young Female       XGB   14.381        13.486        2.412 0.8947          OK
  Middle-aged Male       XGB   25.932        26.571        1.505 0.6389          OK
Middle-aged Female       XGB   21.189        20.087        1.352 1.1017     Caution
      Elderly Male        RF   26.296        25.177        0.938 1.1195     Caution
    Elderly Female       XGB   24.417        23.408        1.283 1.0087     Caution

── G2-b: temporal validation ──
             Group Algorithm  HO_RMSE  Temporal_RMSE  Delta   Direction G2b_verdict
        Young Male      LGBM    7.760         20.611 12.852 degradation        Risk
      Young Female       XGB    7.535         11.176  3.642 degradation        Risk
  Middle-aged Male       XGB   18.950         25.274  6.324 degradation        Risk
Middle-aged Femal

2026-09-01 15:11:39,099 | WARNING | SHAP failed for Young_Female (XGB): could not convert string to float: '[9.064452E1]'
2026-09-01 15:11:39,151 | WARNING | SHAP failed for Middle_Male (XGB): could not convert string to float: '[1.0748756E2]'
2026-09-01 15:11:39,201 | WARNING | SHAP failed for Middle_Female (XGB): could not convert string to float: '[9.943434E1]'
2026-09-01 15:11:39,591 | WARNING | SHAP failed for Elderly_Female (XGB): could not convert string to float: '[1.0548868E2]'
2026-09-01 15:11:39,608 | INFO | G3 score = 0.050 (mean HHI = 0.282)
2026-09-01 15:11:39,624 | INFO | Composite (G1-corrected) = 0.444 [Red — Do not deploy]
2026-09-01 15:11:39,627 | INFO | Composite (G1-raw)       = 0.224 [Red — Do not deploy]



── G3: Transparency (SHAP HHI) with diffusion-without-accuracy caveat ──
             Group Algorithm  SHAP_HHI  CI_95_lower  CI_95_upper G3_verdict  BalancedAcc Caveat
        Young Male      LGBM    0.1803       0.1750       0.1865    Caution        0.407       
      Young Female       XGB       NaN          NaN          NaN        n/a        0.402       
  Middle-aged Male       XGB       NaN          NaN          NaN        n/a        0.380       
Middle-aged Female       XGB       NaN          NaN          NaN        n/a        0.437       
      Elderly Male        RF    0.3835       0.3611       0.4065       Risk        0.341       
    Elderly Female       XGB       NaN          NaN          NaN        n/a        0.362       

── G4: Accountability — Model Card 10/13 = 0.77 [Caution] ──
  [x] Intended purpose documented
  [x] Training data source and period stated
  [x] Medicated-diabetic exclusion criterion stated
  [x] Performance metrics reported (RMSE/MAE/R2)
  [x] Base-r

2026-09-01 15:11:43,806 | INFO | Figures saved (png + pdf, dpi=600) to results/figures/
2026-09-01 15:11:43,833 | INFO | Notebook 02 complete. Composite (corrected) = 0.444


In [5]:
# ═══════════════════════════════════════════════════════════════════════════════
# NOTEBOOK 02 — AI Governance Evaluation (G1–G4)   (REVISED, self-contained)
# ═══════════════════════════════════════════════════════════════════════════════
# Loads only the primitive artefacts saved by Notebook 01 (df_final, best_models,
# best_algo_name, all_results, config) and computes CV stability, temporal
# validation, SHAP+HHI, and the four-pillar scorecard internally — so the
# governance logic is fully reproducible from this notebook alone.
#
# Revisions (addressing referee comments):
#   [G1-DUAL]   G1 Fairness reported two ways — G1-raw (ΔStageAcc+ΔRMSE, base-rate
#               confounded) and G1-corrected (ΔBalancedAcc+ΔCohenKappa, invariant).
#   [G3-CAVEAT] HHI reported but flagged: low HHI ≠ better transparency.
#   [WEIGHTS]   Weights exposed; one-line sensitivity sweep included.
#   [PATHS]     results/{tables,figures,artifacts}; no absolute paths printed.
#   [FIGURES]   seaborn grayscale; no captions; dpi=600; saved as BOTH png + pdf.
# ═══════════════════════════════════════════════════════════════════════════════

import warnings; warnings.filterwarnings("ignore")
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import shap
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (mean_squared_error, mean_absolute_error, r2_score,
                             balanced_accuracy_score, cohen_kappa_score, f1_score)
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
import lightgbm as lgb
import xgboost as xgb

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger(__name__)

SEED = 42; DPI = 600
np.random.seed(SEED)

# ── Grayscale publication style (seaborn) ─────────────────────────────────────
sns.set_theme(style="whitegrid", context="paper")
sns.set_palette("Greys")
plt.rcParams.update({
    "font.family":"DejaVu Sans", "font.size":11, "axes.unicode_minus":False,
    "figure.dpi":150, "savefig.dpi":DPI,
    "axes.edgecolor":"0.2", "axes.linewidth":0.8,
    "grid.color":"0.85", "grid.linewidth":0.6,
})

def save_fig(fig, name):
    """Save a figure as BOTH png and pdf at dpi=600, no caption, tight bbox."""
    for ext in ("png", "pdf"):
        fig.savefig(FIG_DIR / f"{name}.{ext}", dpi=DPI, bbox_inches="tight")
    plt.close(fig)

# ── Project paths (robust; no absolute path printed) ──────────────────────────
def find_project_root(start: Path = Path.cwd()) -> Path:
    for p in [start, *start.parents]:
        if (p / "data").is_dir() and (p / "results").is_dir():
            return p
    return start.parent if start.name == "notebooks" else start

ROOT      = find_project_root()
FIG_DIR   = ROOT / "results" / "figures"
TABLE_DIR = ROOT / "results" / "tables"
ART_DIR   = ROOT / "results" / "artifacts"
for d in (FIG_DIR, TABLE_DIR, ART_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# 1. LOAD PRIMITIVES FROM NOTEBOOK 01
# ═══════════════════════════════════════════════════════════════════════════════
def _load_df_final():
    p_parq = ART_DIR / "df_final.parquet"; p_csv = ART_DIR / "df_final.csv"
    if p_parq.exists():
        try:    return pd.read_parquet(p_parq)
        except Exception: pass
    return pd.read_csv(p_csv)

df_final       = _load_df_final()
best_models    = joblib.load(ART_DIR / "best_models.pkl")
best_algo_name = joblib.load(ART_DIR / "best_algo_name.pkl")
all_results    = joblib.load(ART_DIR / "all_results.pkl")
cfg            = joblib.load(ART_DIR / "config.pkl")

X_FEATURES   = cfg["X_FEATURES"]; NUM_FEATURES = cfg["NUM_FEATURES"]
CAT_FEATURES = cfg["CAT_FEATURES"]
GROUP_CONFIG = cfg["GROUP_CONFIG"]; GROUP_LABELS = cfg["GROUP_LABELS"]
STAGE_ORDER  = cfg["STAGE_ORDER"]
ALGORITHMS   = ["LR","Ridge","RF","LGBM","XGB","MLP"]

# Guard: after a CSV round-trip some numeric columns can load as object dtype,
# which later breaks XGBoost's SHAP explainer. Coerce all model inputs + FPG to
# float once, up front. (GlucoseStage/text columns are left untouched.)
_num_cols = [c for c in (X_FEATURES + ["FPG", "AgeGroup", "Sex", "SurveyYear"])
             if c in df_final.columns]
df_final[_num_cols] = df_final[_num_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0).astype("float64")
log.info("Coerced %d numeric columns to float64.", len(_num_cols))

def assign_glucose_stage(fpg):
    if fpg < 100:  return "normal"
    if fpg < 126:  return "ifg"
    return "diabetes"
def stage_of(vals):
    return pd.Series([assign_glucose_stage(v) for v in np.asarray(vals)], dtype=object)

log.info("Primitives loaded. df_final: %s", df_final.shape)

def group_holdout(grp):
    c = GROUP_CONFIG[grp]
    df_g = df_final[(df_final["AgeGroup"]==c["age_group"]) &
                    (df_final["Sex"]==c["sex_code"])].copy()
    X, y = df_g[X_FEATURES], df_g["FPG"]
    Xtr, Xv, ytr, yv = train_test_split(X, y, test_size=0.20, random_state=SEED)
    return df_g, Xtr, Xv, ytr, yv

# ═══════════════════════════════════════════════════════════════════════════════
# 2. THRESHOLDS, WEIGHTS & SCORING UTILITIES
# ═══════════════════════════════════════════════════════════════════════════════
THRESHOLDS = {
    "G1_delta_rmse":     {"ok":3.0,  "caution":6.0,  "cite":"Obermeyer et al. (2019)"},
    "G1_delta_stageacc": {"ok":0.10, "caution":0.20, "cite":"Chouldechova (2017), raw"},
    "G1_delta_balacc":   {"ok":0.10, "caution":0.20, "cite":"Hardt et al. (2016), balanced"},
    "G1_delta_kappa":    {"ok":0.15, "caution":0.30, "cite":"Cohen (1960); chance-corrected"},
    "G2_cv_gap":         {"ok":1.0,  "caution":2.0,  "cite":"Varma & Simon (2006)"},
    "G2_temporal":       {"ok":1.5,  "caution":3.5,  "cite":"Nestor et al. (2019)"},
    "G2_perturbation":   {"ok":1.0,  "caution":2.5,  "cite":"Ghorbani & Zou (2019)"},
    "G3_hhi":            {"ok":0.18, "caution":0.25, "cite":"Lundberg et al. (2020)"},
    "G4_model_card":     {"ok":0.90, "caution":0.70, "cite":"Mitchell et al. (2019)"},
}
WEIGHTS = {"g1":0.35, "g2":0.25, "g3":0.20, "g4":0.20}

def verdict(value, key, low_is_good=True):
    t = THRESHOLDS[key]
    if low_is_good:
        if value <= t["ok"]:      return "OK"
        if value <= t["caution"]: return "Caution"
        return "Risk"
    else:
        if value >= t["ok"]:      return "OK"
        if value >= t["caution"]: return "Caution"
        return "Risk"

def normalise(value, ok_val, risk_val, low_is_good=True, floor=0.05):
    if low_is_good:
        if value <= ok_val:   return 1.0
        if value >= risk_val: return floor
        return floor + (1.0-floor)*(1.0-(value-ok_val)/(risk_val-ok_val))
    else:
        if value >= ok_val:   return 1.0
        if value <= risk_val: return floor
        return floor + (1.0-floor)*((value-risk_val)/(ok_val-risk_val))

log.info("Thresholds, weights, utilities defined.")

# ═══════════════════════════════════════════════════════════════════════════════
# 3. G1 — FAIRNESS  (dual: raw + base-rate-corrected)   ★ core revision
# ═══════════════════════════════════════════════════════════════════════════════
g1_rows = []
for grp in GROUP_CONFIG:
    model = best_models.get(grp)
    if model is None: continue
    _, _, Xv, _, yv = group_holdout(grp)
    yp = model.predict(Xv)

    rmse = float(np.sqrt(mean_squared_error(yv, yp)))
    mae  = float(mean_absolute_error(yv, yp))
    r2   = float(r2_score(yv, yp))
    bias = float(np.mean(yp - yv.values))

    st, sp = stage_of(yv.values), stage_of(yp)
    stage_acc = float((st.values==sp.values).mean())
    maj_cls   = st.value_counts().idxmax()
    maj_base  = float((st.values==maj_cls).mean())
    bal_acc   = float(balanced_accuracy_score(st, sp))
    kappa     = float(cohen_kappa_score(st, sp, labels=STAGE_ORDER))
    macro_f1  = float(f1_score(st, sp, average="macro", labels=STAGE_ORDER))

    g1_rows.append({"Group":GROUP_LABELS[grp], "_key":grp, "Algorithm":best_algo_name[grp],
        "n_test":int(len(yv)), "RMSE":round(rmse,3), "MAE":round(mae,3), "R2":round(r2,3),
        "Bias":round(bias,3), "StageAcc":round(stage_acc,3), "MajBaseline":round(maj_base,3),
        "Improvement":round(stage_acc-maj_base,3), "BalancedAcc":round(bal_acc,3),
        "CohenKappa":round(kappa,3), "MacroF1":round(macro_f1,3)})

g1_df = pd.DataFrame(g1_rows)

d_rmse     = round(g1_df["RMSE"].max()        - g1_df["RMSE"].min(), 3)
d_stageacc = round(g1_df["StageAcc"].max()    - g1_df["StageAcc"].min(), 3)
d_balacc   = round(g1_df["BalancedAcc"].max() - g1_df["BalancedAcc"].min(), 3)
d_kappa    = round(g1_df["CohenKappa"].max()  - g1_df["CohenKappa"].min(), 3)
d_improve  = round(g1_df["Improvement"].max() - g1_df["Improvement"].min(), 3)

g1_summary = pd.DataFrame([
    {"Metric":"Δ-RMSE",            "Value":d_rmse,     "Verdict":verdict(d_rmse,"G1_delta_rmse"),     "Type":"raw"},
    {"Metric":"Δ-StageAccuracy",   "Value":d_stageacc, "Verdict":verdict(d_stageacc,"G1_delta_stageacc"),"Type":"raw (confounded)"},
    {"Metric":"Δ-BalancedAccuracy","Value":d_balacc,   "Verdict":verdict(d_balacc,"G1_delta_balacc"),  "Type":"corrected"},
    {"Metric":"Δ-CohenKappa",      "Value":d_kappa,    "Verdict":verdict(d_kappa,"G1_delta_kappa"),    "Type":"corrected"},
    {"Metric":"Δ-Improvement",     "Value":d_improve,  "Verdict":"—",                                  "Type":"corrected (context)"},
])

print("\n── G1 per-group metrics ──")
print(g1_df.drop(columns=["_key"]).to_string(index=False))
print("\n── G1 disparities: raw vs base-rate-corrected ──")
print(g1_summary.to_string(index=False))

g1_df.drop(columns=["_key"]).to_csv(TABLE_DIR / "table_g1_fairness.csv", index=False, encoding="utf-8")
g1_summary.to_csv(TABLE_DIR / "table_g1_summary.csv", index=False, encoding="utf-8")

s_g1_raw       = round((normalise(d_rmse,3.0,6.0) + normalise(d_stageacc,0.10,0.20))/2, 4)
s_g1_corrected = round((normalise(d_balacc,0.10,0.20) + normalise(d_kappa,0.15,0.30))/2, 4)
log.info("G1-raw score = %.3f | G1-corrected score = %.3f", s_g1_raw, s_g1_corrected)

# ═══════════════════════════════════════════════════════════════════════════════
# 4. G2 — ROBUSTNESS  (CV stability + temporal + perturbation)
# ═══════════════════════════════════════════════════════════════════════════════
def refit(algo, Xtr, ytr):
    if   algo=="LR":    m = LinearRegression()
    elif algo=="Ridge": m = Ridge(random_state=SEED)
    elif algo=="RF":    m = RandomForestRegressor(n_estimators=300, max_depth=8, random_state=SEED, n_jobs=-1)
    elif algo=="LGBM":  m = lgb.LGBMRegressor(n_estimators=300, max_depth=5, learning_rate=0.05, random_state=SEED, n_jobs=-1, verbose=-1)
    elif algo=="XGB":   m = xgb.XGBRegressor(n_estimators=300, max_depth=5, learning_rate=0.05, random_state=SEED, tree_method="hist", verbosity=0)
    elif algo=="MLP":   m = MLPRegressor(hidden_layer_sizes=(128,128), max_iter=300, random_state=SEED)
    m.fit(Xtr, ytr); return m

# G2-a: 5-fold CV stability
g2a_rows = []
for grp in GROUP_CONFIG:
    algo = best_algo_name[grp]
    df_g, Xtr, Xv, ytr, yv = group_holdout(grp)
    ho_rmse = float(np.sqrt(mean_squared_error(yv, best_models[grp].predict(Xv))))
    X, y = df_g[X_FEATURES], df_g["FPG"]
    kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
    fold_rmses = []
    for tr, va in kf.split(X):
        m = refit(algo, X.iloc[tr], y.iloc[tr])
        fold_rmses.append(np.sqrt(mean_squared_error(y.iloc[va], m.predict(X.iloc[va]))))
    cv_mean, cv_std = float(np.mean(fold_rmses)), float(np.std(fold_rmses))
    gap = round(abs(cv_mean - ho_rmse), 4)
    g2a_rows.append({"Group":GROUP_LABELS[grp], "Algorithm":algo,
        "HO_RMSE":round(ho_rmse,3), "CV_RMSE_mean":round(cv_mean,3),
        "CV_RMSE_std":round(cv_std,3), "Gap":gap, "G2a_verdict":verdict(gap,"G2_cv_gap")})
g2a_df = pd.DataFrame(g2a_rows)

# G2-b: temporal validation (train earlier cycles → test most recent)
years = sorted(pd.to_numeric(df_final["SurveyYear"], errors="coerce").dropna().unique())
YEAR_TEST  = years[-1] if years else None
YEAR_TRAIN = years[:-1] if len(years) > 1 else years
g2b_rows = []
if YEAR_TEST is not None and len(YEAR_TRAIN) > 0:
    for grp in GROUP_CONFIG:
        algo = best_algo_name[grp]; c = GROUP_CONFIG[grp]
        df_g = df_final[(df_final["AgeGroup"]==c["age_group"]) & (df_final["Sex"]==c["sex_code"])]
        tr = df_g[df_g["SurveyYear"].isin(YEAR_TRAIN)]
        te = df_g[df_g["SurveyYear"]==YEAR_TEST]
        if len(tr) < 50 or len(te) < 20:
            continue
        m = refit(algo, tr[X_FEATURES], tr["FPG"])
        _, _, Xv_g, _, yv_g = group_holdout(grp)
        ho_rmse = float(np.sqrt(mean_squared_error(yv_g, m.predict(Xv_g))))
        te_rmse = float(np.sqrt(mean_squared_error(te["FPG"], m.predict(te[X_FEATURES]))))
        delta = round(te_rmse - ho_rmse, 3)
        g2b_rows.append({"Group":GROUP_LABELS[grp], "Algorithm":algo,
            "HO_RMSE":round(ho_rmse,3), "Temporal_RMSE":round(te_rmse,3), "Delta":delta,
            "Direction":("degradation" if delta>0 else "stable/improve"),
            "G2b_verdict":verdict(max(delta,0.0),"G2_temporal")})
g2b_df = pd.DataFrame(g2b_rows)

# G2-c: feature perturbation (±10% Gaussian noise, 5 repeats)
g2c_rows = []; rng = np.random.RandomState(SEED)
for grp in GROUP_CONFIG:
    model = best_models.get(grp)
    if model is None: continue
    _, _, Xv, _, yv = group_holdout(grp)
    base = float(np.sqrt(mean_squared_error(yv, model.predict(Xv))))
    drops = []
    for _ in range(5):
        Xn = Xv.copy()
        for col in Xn.columns:
            s = Xn[col].std()
            if s > 0: Xn[col] += rng.normal(0, s*0.10, len(Xn))
        drops.append(abs(float(np.sqrt(mean_squared_error(yv, model.predict(Xn)))) - base))
    md = round(float(np.mean(drops)), 4)
    g2c_rows.append({"Group":GROUP_LABELS[grp], "Algorithm":best_algo_name[grp],
        "RMSE_base":round(base,3), "RMSE_drop_mean":md, "RMSE_drop_std":round(float(np.std(drops)),4),
        "G2c_verdict":verdict(md,"G2_perturbation")})
g2c_df = pd.DataFrame(g2c_rows)

print("\n── G2-a: CV stability ──");        print(g2a_df.to_string(index=False))
print("\n── G2-b: temporal validation ──"); print(g2b_df.to_string(index=False) if len(g2b_df) else "  (insufficient per-year n)")
print("\n── G2-c: perturbation ──");         print(g2c_df.to_string(index=False))

pd.concat([g2a_df.assign(dimension="G2a_CV"),
           g2b_df.assign(dimension="G2b_Temporal") if len(g2b_df) else pd.DataFrame(),
           g2c_df.assign(dimension="G2c_Perturbation")], ignore_index=True
          ).to_csv(TABLE_DIR / "table_g2_robustness.csv", index=False, encoding="utf-8")

s_g2a = normalise(g2a_df["Gap"].mean(), 1.0, 2.0)
s_g2c = normalise(g2c_df["RMSE_drop_mean"].mean(), 1.0, 2.5)
if len(g2b_df):
    deg = g2b_df.loc[g2b_df["Delta"]>0, "Delta"]
    s_g2b = normalise(deg.max() if len(deg) else 0.0, 1.5, 3.5)
else:
    s_g2b = 1.0
s_g2 = round((s_g2a + s_g2b + s_g2c)/3, 4)
log.info("G2 score = %.3f (a=%.2f b=%.2f c=%.2f)", s_g2, s_g2a, s_g2b, s_g2c)

# ═══════════════════════════════════════════════════════════════════════════════
# 5. G3 — TRANSPARENCY  (SHAP HHI, bootstrapped) + explicit caveat  [G3-CAVEAT]
# ═══════════════════════════════════════════════════════════════════════════════
def compute_hhi_bootstrap(shap_vals, n_boot=1000, ci=0.95):
    def _hhi(sv):
        m = np.mean(np.abs(sv), axis=0); tot = m.sum()
        if tot == 0: return 0.0
        s = m/tot; return float(np.sum(s**2))
    point = _hhi(shap_vals)
    rng_ = np.random.RandomState(SEED); n = shap_vals.shape[0]; boots = []
    for _ in range(n_boot):
        idx = rng_.randint(0, n, n)
        boots.append(_hhi(shap_vals[idx]))
    lo = float(np.percentile(boots, (1-ci)/2*100))
    hi = float(np.percentile(boots, (1+ci)/2*100))
    return {"HHI":round(point,4), "CI_lo":round(lo,4), "CI_hi":round(hi,4)}

def shap_values_native(model, algo, Xdf):
    """Compute SHAP values via each library's NATIVE path.
    Rationale: shap 0.49.x fails to parse XGBoost 3.x tree dumps
    ('could not convert string to float: [..E1]'). XGBoost/LightGBM can emit
    exact SHAP contributions directly, bypassing the shap-library parser."""
    Xf = Xdf.apply(pd.to_numeric, errors="coerce").fillna(0.0).astype("float64")
    if algo == "XGB":
        contribs = model.get_booster().predict(xgb.DMatrix(Xf), pred_contribs=True)
        return np.asarray(contribs)[:, :-1]          # drop trailing bias column
    if algo == "LGBM":
        contribs = model.predict(Xf, pred_contrib=True)
        return np.asarray(contribs)[:, :-1]          # drop trailing bias column
    if algo == "RF":
        sv = shap.TreeExplainer(model).shap_values(Xf)
        sv = np.asarray(sv)
        if sv.ndim == 3:
            sv = sv[..., 0] if sv.shape[-1] == 1 else sv.reshape(sv.shape[0], -1)
        return sv
    # non-tree fallback (KernelExplainer)
    bg = shap.sample(Xf, min(50, len(Xf)), random_state=SEED)
    sv = shap.KernelExplainer(model.predict, bg).shap_values(
        Xf.iloc[:min(100, len(Xf))], nsamples=100)
    return np.asarray(sv)

g3_rows = []; shap_store = {}
for grp in GROUP_CONFIG:
    model = best_models.get(grp); algo = best_algo_name.get(grp)
    if model is None: continue
    _, _, Xv, _, _ = group_holdout(grp)
    Xv_num = Xv.apply(pd.to_numeric, errors="coerce").fillna(0.0).astype("float64")

    hhi = {"HHI": np.nan, "CI_lo": np.nan, "CI_hi": np.nan}
    try:
        sv = shap_values_native(model, algo, Xv_num)
        shap_store[grp] = {"shap": sv, "X": Xv_num.iloc[:sv.shape[0]].copy()}
        hhi = compute_hhi_bootstrap(sv)
    except Exception as e:
        log.warning("SHAP failed for %s (%s): %s", grp, algo, e)

    g3_rows.append({"Group":GROUP_LABELS[grp], "Algorithm":algo,
        "SHAP_HHI":hhi["HHI"], "CI_95_lower":hhi["CI_lo"], "CI_95_upper":hhi["CI_hi"],
        "G3_verdict":verdict(hhi["HHI"],"G3_hhi") if not np.isnan(hhi["HHI"]) else "n/a"})
g3_df = pd.DataFrame(g3_rows)

g3_df = g3_df.merge(g1_df[["Group","BalancedAcc"]], on="Group", how="left")
g3_df["Caveat"] = np.where(
    (g3_df["SHAP_HHI"] < 0.18) & (g3_df["BalancedAcc"] < 0.42),
    "diffuse attribution but low accuracy — not genuine transparency", "")
print("\n── G3: Transparency (SHAP HHI) with diffusion-without-accuracy caveat ──")
print(g3_df.to_string(index=False))
g3_df.to_csv(TABLE_DIR / "table_g3_transparency.csv", index=False, encoding="utf-8")

s_g3 = normalise(g3_df["SHAP_HHI"].mean(skipna=True), 0.18, 0.25)
log.info("G3 score = %.3f (mean HHI = %.3f)", s_g3, g3_df["SHAP_HHI"].mean(skipna=True))

# ═══════════════════════════════════════════════════════════════════════════════
# 6. G4 — ACCOUNTABILITY (Model Card checklist)
# ═══════════════════════════════════════════════════════════════════════════════
GITHUB_URL = ""   # set to public repo URL when available
MODEL_CARD = {
    "Intended purpose documented": True,
    "Training data source and period stated": True,
    "Medicated-diabetic exclusion criterion stated": True,
    "Performance metrics reported (RMSE/MAE/R2)": True,
    "Base-rate-corrected fairness metrics reported": True,
    "Limitations and failure modes disclosed": True,
    "Disaggregated performance by group": True,
    "SHAP attribution available": True,
    "DiCE counterfactual paths provided": True,
    "LLM personalised reports generated": True,
    "Code publicly available (GitHub)": bool(GITHUB_URL),
    "Human-in-the-loop procedure defined": False,   # gap
    "Privacy Impact Assessment (PIA) conducted": False,   # gap
}
total = len(MODEL_CARD); fulfilled = sum(MODEL_CARD.values())
mc_score = round(fulfilled/total, 4)
mc_verdict = verdict(mc_score, "G4_model_card", low_is_good=False)
print(f"\n── G4: Accountability — Model Card {fulfilled}/{total} = {mc_score:.2f} [{mc_verdict}] ──")
for k, v in MODEL_CARD.items():
    print(f"  [{'x' if v else ' '}] {k}")
s_g4 = normalise(mc_score, 0.90, 0.70, low_is_good=False)
pd.DataFrame([{"item":k, "fulfilled":v} for k,v in MODEL_CARD.items()]
             ).to_csv(TABLE_DIR / "table_g4_accountability.csv", index=False, encoding="utf-8")

# ═══════════════════════════════════════════════════════════════════════════════
# 7. COMPOSITE SCORECARD  (default: G1-corrected; G1-raw reported alongside)
# ═══════════════════════════════════════════════════════════════════════════════
def composite_of(s_g1):
    return round(WEIGHTS["g1"]*s_g1 + WEIGHTS["g2"]*s_g2 + WEIGHTS["g3"]*s_g3 + WEIGHTS["g4"]*s_g4, 4)
comp_corrected = composite_of(s_g1_corrected)
comp_raw       = composite_of(s_g1_raw)
def grade(c):
    return ("Green — Deploy" if c>=0.80 else "Amber — Conditional" if c>=0.60 else "Red — Do not deploy")

scorecard = pd.DataFrame([
    {"Pillar":"G1 Fairness (corrected)","Weight":WEIGHTS["g1"],"Score":s_g1_corrected,
     "Verdict":verdict(d_balacc,"G1_delta_balacc")},
    {"Pillar":"  G1 Fairness (raw, ref.)","Weight":"—","Score":s_g1_raw,
     "Verdict":verdict(d_stageacc,"G1_delta_stageacc")},
    {"Pillar":"G2 Robustness","Weight":WEIGHTS["g2"],"Score":s_g2,
     "Verdict":"OK" if s_g2>=0.8 else "Caution" if s_g2>=0.5 else "Risk"},
    {"Pillar":"G3 Transparency","Weight":WEIGHTS["g3"],"Score":s_g3,
     "Verdict":g3_df["G3_verdict"].mode()[0] if len(g3_df) else "n/a"},
    {"Pillar":"G4 Accountability","Weight":WEIGHTS["g4"],"Score":s_g4,"Verdict":mc_verdict},
    {"Pillar":"COMPOSITE (G1-corrected)","Weight":1.00,"Score":comp_corrected,"Verdict":grade(comp_corrected)},
    {"Pillar":"COMPOSITE (G1-raw, ref.)","Weight":"—","Score":comp_raw,"Verdict":grade(comp_raw)},
])
print("\n── Four-pillar governance scorecard ──")
print(scorecard.to_string(index=False))
scorecard.to_csv(TABLE_DIR / "table_governance_scorecard.csv", index=False, encoding="utf-8")
log.info("Composite (G1-corrected) = %.3f [%s]", comp_corrected, grade(comp_corrected))
log.info("Composite (G1-raw)       = %.3f [%s]", comp_raw, grade(comp_raw))

# Weight sensitivity sweep
sens_rows = []
for w1 in [0.25, 0.30, 0.35, 0.40, 0.45]:
    rem = 1 - w1
    w2, w3, w4 = rem*0.25/0.65, rem*0.20/0.65, rem*0.20/0.65
    c = round(w1*s_g1_corrected + w2*s_g2 + w3*s_g3 + w4*s_g4, 4)
    sens_rows.append({"w_G1":w1, "Composite_corrected":c, "Grade":grade(c)})
sens_df = pd.DataFrame(sens_rows)
print("\n── Weight sensitivity (G1 weight swept) ──")
print(sens_df.to_string(index=False))
sens_df.to_csv(TABLE_DIR / "table_weight_sensitivity.csv", index=False, encoding="utf-8")

# ═══════════════════════════════════════════════════════════════════════════════
# 8. FIGURES  (seaborn grayscale; no captions; dpi=600; png + pdf)
# ═══════════════════════════════════════════════════════════════════════════════
labels = [GROUP_LABELS[g] for g in GROUP_CONFIG]

# --- Figure 1: G1 raw vs corrected (the key referee-facing plot) ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# (a) StageAcc vs Majority baseline — long-form for seaborn
sa = [g1_df.loc[g1_df["Group"]==l,"StageAcc"].values[0] for l in labels]
mb = [g1_df.loc[g1_df["Group"]==l,"MajBaseline"].values[0] for l in labels]
dfa = pd.DataFrame({"Group":labels*2,
                    "Value":sa+mb,
                    "Metric":["Model Stage Accuracy"]*len(labels)+["Majority baseline"]*len(labels)})
sns.barplot(data=dfa, x="Group", y="Value", hue="Metric",
            palette=["0.30","0.75"], edgecolor="black", ax=axes[0])
axes[0].set_ylim(0,1); axes[0].set_ylabel("Accuracy"); axes[0].set_xlabel("")
axes[0].set_xticklabels(labels, rotation=30, ha="right", fontsize=8)
axes[0].legend(fontsize=8, loc="lower left", title="")

# (b) Balanced accuracy vs Cohen's kappa
ba = [g1_df.loc[g1_df["Group"]==l,"BalancedAcc"].values[0] for l in labels]
kp = [g1_df.loc[g1_df["Group"]==l,"CohenKappa"].values[0] for l in labels]
dfb = pd.DataFrame({"Group":labels*2,
                    "Value":ba+kp,
                    "Metric":["Balanced accuracy"]*len(labels)+["Cohen's kappa"]*len(labels)})
sns.barplot(data=dfb, x="Group", y="Value", hue="Metric",
            palette=["0.30","0.75"], edgecolor="black", ax=axes[1])
axes[1].set_ylim(0,1); axes[1].set_ylabel("Score"); axes[1].set_xlabel("")
axes[1].set_xticklabels(labels, rotation=30, ha="right", fontsize=8)
axes[1].legend(fontsize=8, loc="upper right", title="")
plt.tight_layout()
save_fig(fig, "fig_g1_raw_vs_corrected")

# --- Figure 2: governance radar (grayscale) ---
cats = ["G1\nFairness","G2\nRobustness","G3\nTransparency","G4\nAccountability"]
scores = [s_g1_corrected, s_g2, s_g3, s_g4]
N = len(cats); ang = np.linspace(0, 2*np.pi, N, endpoint=False).tolist(); ang += ang[:1]
sc = scores + [scores[0]]
fig, ax = plt.subplots(figsize=(7,7), subplot_kw=dict(polar=True))
ax.plot(ang, sc, "o-", color="0.15", lw=2.5, ms=8)
ax.fill(ang, sc, color="0.15", alpha=0.15)
ax.plot(ang, [0.80]*(N+1), ls="--", color="0.45", lw=1.2, label="Deploy threshold (0.80)")
ax.plot(ang, [0.60]*(N+1), ls=":",  color="0.45", lw=1.2, label="Conditional threshold (0.60)")
ax.set_xticks(ang[:-1]); ax.set_xticklabels(cats, fontsize=11, fontweight="bold")
ax.set_ylim(0,1); ax.set_yticks([0.2,0.4,0.6,0.8,1.0])
ax.set_yticklabels(["0.2","0.4","0.6","0.8","1.0"], fontsize=8, color="0.4")
for a, s in zip(ang[:-1], scores):
    ax.text(a, s+0.09, f"{s:.2f}", ha="center", va="center", fontsize=10, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="0.3", alpha=0.9))
ax.legend(fontsize=9, loc="lower right", bbox_to_anchor=(1.25,-0.08))
plt.tight_layout(pad=2.5)
save_fig(fig, "fig_governance_radar")

# --- Figure 3: RMSE heatmap (seaborn grayscale) ---
rmse_mat = pd.DataFrame(index=list(GROUP_CONFIG), columns=ALGORITHMS, dtype=float)
for grp in GROUP_CONFIG:
    for algo in ALGORITHMS:
        m = all_results.get(grp, {}).get(algo)
        rmse_mat.loc[grp, algo] = m["RMSE"] if m else np.nan
rmse_mat.index = [GROUP_LABELS[g] for g in rmse_mat.index]
fig, ax = plt.subplots(figsize=(11,5))
sns.heatmap(rmse_mat.astype(float), annot=True, fmt=".2f", cmap="Greys",
            linewidths=0.5, linecolor="white",
            cbar_kws={"label":"RMSE (mg/dL)", "shrink":0.8}, ax=ax)
for i, grp in enumerate(GROUP_CONFIG):
    ba_ = best_algo_name.get(grp)
    if ba_ in ALGORITHMS:
        ax.text(ALGORITHMS.index(ba_)+0.5, i+0.22, "*", ha="center", va="center",
                fontsize=16, color="black")
ax.set_xlabel("Algorithm"); ax.set_ylabel("Demographic Group")
plt.tight_layout()
save_fig(fig, "fig_rmse_heatmap")

log.info("Figures saved (png + pdf, dpi=%d) to results/figures/", DPI)

# ═══════════════════════════════════════════════════════════════════════════════
# 9. SAVE CONSOLIDATED GOVERNANCE ARTEFACTS
# ═══════════════════════════════════════════════════════════════════════════════
joblib.dump({"g1_df":g1_df, "g1_summary":g1_summary,
             "s_g1_raw":s_g1_raw, "s_g1_corrected":s_g1_corrected,
             "s_g2":s_g2, "s_g3":s_g3, "s_g4":s_g4,
             "comp_corrected":comp_corrected, "comp_raw":comp_raw,
             "g2a_df":g2a_df, "g2b_df":g2b_df, "g2c_df":g2c_df,
             "g3_df":g3_df, "shap_store":shap_store, "scorecard":scorecard,
             "sens_df":sens_df, "WEIGHTS":WEIGHTS, "THRESHOLDS":THRESHOLDS},
            ART_DIR / "governance_results.pkl")
log.info("Notebook 02 complete. Composite (corrected) = %.3f", comp_corrected)

2026-09-01 15:18:53,653 | INFO | Coerced 35 numeric columns to float64.
2026-09-01 15:18:53,653 | INFO | Primitives loaded. df_final: (27934, 36)
2026-09-01 15:18:53,655 | INFO | Thresholds, weights, utilities defined.
2026-09-01 15:18:53,969 | INFO | G1-raw score = 0.050 | G1-corrected score = 0.680



── G1 per-group metrics ──
             Group Algorithm  n_test   RMSE    MAE    R2   Bias  StageAcc  MajBaseline  Improvement  BalancedAcc  CohenKappa  MacroF1
        Young Male      LGBM     613 11.239  7.017 0.091  0.922     0.790        0.783        0.007        0.407       0.259    0.415
      Young Female       XGB     719 14.381  7.164 0.159 -0.530     0.880        0.890       -0.010        0.402       0.256    0.411
  Middle-aged Male       XGB    1096 25.932 15.331 0.061 -0.246     0.462        0.483       -0.021        0.380       0.102    0.307
Middle-aged Female       XGB    1461 21.189 10.977 0.119  0.002     0.625        0.656       -0.031        0.437       0.248    0.424
      Elderly Male        RF     739 26.296 16.589 0.019 -0.486     0.441        0.438        0.003        0.341       0.008    0.220
    Elderly Female       XGB     961 24.417 15.526 0.043 -0.955     0.417        0.504       -0.086        0.362       0.058    0.280

── G1 disparities: raw vs base-ra

2026-09-01 15:19:34,289 | INFO | G2 score = 0.481 (a=0.39 b=0.05 c=1.00)



── G2-a: CV stability ──
             Group Algorithm  HO_RMSE  CV_RMSE_mean  CV_RMSE_std    Gap G2a_verdict
        Young Male      LGBM   11.239        16.309        3.305 5.0699        Risk
      Young Female       XGB   14.381        13.486        2.412 0.8947          OK
  Middle-aged Male       XGB   25.932        26.571        1.505 0.6389          OK
Middle-aged Female       XGB   21.189        20.087        1.352 1.1017     Caution
      Elderly Male        RF   26.296        25.177        0.938 1.1195     Caution
    Elderly Female       XGB   24.417        23.408        1.283 1.0087     Caution

── G2-b: temporal validation ──
             Group Algorithm  HO_RMSE  Temporal_RMSE  Delta   Direction G2b_verdict
        Young Male      LGBM    7.760         20.611 12.852 degradation        Risk
      Young Female       XGB    7.535         11.176  3.642 degradation        Risk
  Middle-aged Male       XGB   18.950         25.274  6.324 degradation        Risk
Middle-aged Femal

2026-09-01 15:19:37,019 | INFO | G3 score = 1.000 (mean HHI = 0.168)
2026-09-01 15:19:37,040 | INFO | Composite (G1-corrected) = 0.634 [Amber — Conditional]
2026-09-01 15:19:37,042 | INFO | Composite (G1-raw)       = 0.413 [Red — Do not deploy]



── G3: Transparency (SHAP HHI) with diffusion-without-accuracy caveat ──
             Group Algorithm  SHAP_HHI  CI_95_lower  CI_95_upper G3_verdict  BalancedAcc                                                          Caveat
        Young Male      LGBM    0.1803       0.1750       0.1865    Caution        0.407                                                                
      Young Female       XGB    0.1577       0.1502       0.1647         OK        0.402 diffuse attribution but low accuracy — not genuine transparency
  Middle-aged Male       XGB    0.0660       0.0640       0.0682         OK        0.380 diffuse attribution but low accuracy — not genuine transparency
Middle-aged Female       XGB    0.1224       0.1186       0.1265         OK        0.437                                                                
      Elderly Male        RF    0.3835       0.3611       0.4065       Risk        0.341                                                                
    Elde

2026-09-01 15:19:41,594 | INFO | Figures saved (png + pdf, dpi=600) to results/figures/
2026-09-01 15:19:41,636 | INFO | Notebook 02 complete. Composite (corrected) = 0.634
